# Introduction

This notebook provides a guide on how to perform MLIP property calculations with MatCalc.

For the purposes of this notebook, we will only use the TensorNet PBE and r2SCAN and MACE-MPA-0 foundation potentials. These requires MatGL and mace-torch to be installed. Uncomment the next cell to install these packages if you do not already have them.

In [ ]:
! uv pip install matcalc "matgl>=2.2.0" mace-torch

Using Python 3.11.15 environment at: /Users/shyue/repos/matcalc/.venv
Checked 3 packages in 66ms


In [ ]:
from __future__ import annotations

import warnings
from time import perf_counter
import pandas as pd

import matplotlib.pyplot as plt
from pymatgen.ext.matproj import MPRester
from tqdm import tqdm
import plotly.express as px

from matcalc import ElasticityCalc, EOSCalc, RelaxCalc, ChainedCalc

warnings.filterwarnings("ignore", category=UserWarning)

# Obtaining Test Structures from the Materials Project

We will first use Pymatgen's MPRester to obtain a few LiCl and NaCl structures from the Materials Project for demostration purposes.

In [ ]:
mp_data = MPRester().materials.summary.search(
    formula=["LiCl", "Li2O", "NaCl"], _fields=["material_id", "structure", "energy_above_hull"]
)
mp_data.sort(key=lambda d: d["energy_above_hull"])
print(f"{len(mp_data)} structures were downloaded")

11 structures were downloaded


# Setup

Let us set some parameters we will be using.

In [ ]:
# This is the list of models we are using for this demo.
models = ["TensorNet-PES-MatPES-PBE-2025.1", "MACE"]

# This is the max force used for EOS and Elasticity calculations.
fmax = 0.05
optimizer = "BFGSLineSearch"
units_GPa = True

# Example 1: Running ElasticityCalc on one structure

In [ ]:
prop_calc = ElasticityCalc("TensorNet-PES-MatPES-PBE-2025.1", relax_structure=True, fmax=fmax, units_GPa=units_GPa)
prop_calc.calc(mp_data[0])

{'material_id': 'mp-1960',
 'structure': Structure Summary
 Lattice
     abc : 2.994747056473457 2.9947469719687705 2.9947467240164025
  angles : 59.999995375890364 59.999995586566556 59.99999708813193
  volume : 18.991765649922772
       A : np.float64(2.5935268877725326) np.float64(1.1598877808384269e-07) np.float64(1.4973737725288703)
       B : np.float64(0.8645087780329525) np.float64(2.445200571149694) np.float64(1.4973737561708844)
       C : np.float64(-5.1402060998567464e-08) np.float64(-5.6381106564677454e-08) np.float64(2.9947467240164016)
     pbc : True True True
 PeriodicSite: Li (2.594, 1.834, 4.492) [0.75, 0.75, 0.75]
 PeriodicSite: Li (0.8645, 0.6113, 1.497) [0.25, 0.25, 0.25]
 PeriodicSite: O (6.492e-08, 1.127e-09, -5.019e-08) [2.488e-08, 4.607e-10, -2.943e-08],
 'energy_above_hull': 0.0,
 'final_structure': Structure Summary
 Lattice
     abc : 2.994747056473457 2.9947469719687705 2.9947467240164025
  angles : 59.999995375890364 59.999995586566556 59.99999708813193
 

A few notes about the args adn return values from a PropCalc.
1. The input can be a structure or a dict containing a structure key.
2. The return value is always a dictionary that contains all the original keys from the input.


## Example 2: Chaining Calculators

Very often, you want to run multiple calculations using the same MLIP. This can be done using ChainedCalc. One thing to note that you typically do not want to waste compute re-relaxing the structure after the first relaxation. So the `relax_structure` parameter for other calculators are set to False.

In [ ]:
propcalc = ChainedCalc([
    RelaxCalc("TensorNet-PES-MatPES-PBE-2025.1", fmax=fmax, optimizer=optimizer), 
    ElasticityCalc("TensorNet-PES-MatPES-PBE-2025.1", fmax=fmax, relax_structure=False), 
    EOSCalc("TensorNet-PES-MatPES-PBE-2025.1", fmax=fmax, optimizer=optimizer, relax_structure=False)])
propcalc.calc(mp_data[0])

{'material_id': 'mp-1960',
 'structure': Structure Summary
 Lattice
     abc : 3.002485482489818 3.0024854053251318 3.002485046766873
  angles : 60.0000014919968 60.000003342549434 60.000005749604355
  volume : 19.139373219715075
       A : np.float64(2.600228677314417) np.float64(-3.980765688504616e-08) np.float64(1.5012427845734453)
       B : np.float64(0.8667425376597968) np.float64(2.4515191241780125) np.float64(1.5012428072736752)
       C : np.float64(-2.2519193055538185e-07) np.float64(-1.314327812199908e-07) np.float64(3.0024850467668616)
     pbc : True True True
 PeriodicSite: Li (2.6, 1.839, 4.504) [0.75, 0.75, 0.75]
 PeriodicSite: Li (0.8667, 0.6129, 1.501) [0.25, 0.25, 0.25]
 PeriodicSite: O (9.769e-08, -5.493e-08, -5.282e-08) [4.504e-08, -2.24e-08, -2.891e-08],
 'energy_above_hull': 0.0,
 'final_structure': Structure Summary
 Lattice
     abc : 3.002485482489818 3.0024854053251318 3.002485046766873
  angles : 60.0000014919968 60.000003342549434 60.000005749604355
  volum

Note that the final result dict contains keys from all calculators, including `final_structure` from `RelaxCalc`, `bulk_modulus_vrh` and `shear_modulus_vrh` from ElasticityCalc, and `eos` and `bulk_modulus_bm` from EOSCalc.

## Example 3: Multiple Properties with Multiple MLIPs

In this final example, we will bring everything together with multiple properties with multiple MLIPs. We will use ChainedCalc from Example 2 to make things easier.

In [ ]:
prop_preds = []

for dct in (pbar := tqdm(mp_data)):
    mat_id, structure = dct["material_id"], dct["structure"]
    formula = structure.reduced_formula
    pbar.set_description(f"Running {mat_id} ({formula})")

    for model in models:
        propcalc = ChainedCalc([
            RelaxCalc(model, fmax=fmax, optimizer=optimizer), 
            ElasticityCalc(model, fmax=fmax, relax_structure=False, units_GPa=True), 
            EOSCalc(model, fmax=fmax, optimizer=optimizer, relax_structure=False)])
        results = propcalc.calc(dct)
        prop_preds.append({
            "material_id": mat_id, 
            "formula": formula,
            "model": model,
        } | {k: results[k] for k in ["final_structure", "bulk_modulus_vrh", "shear_modulus_vrh", "bulk_modulus_bm", "eos"]}
                         )


Running mp-1960 (Li2O):   0%|                                                                                                | 0/11 [00:00<?, ?it/s]WARNING:root:Default dtype float32 does not match model dtype float64, converting models to float32.


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.
Using medium MPA-0 model as default MACE-MP model, to use previous (before 3.10) default model please specify 'medium' as model argument
Using Materials Project MACE for MACECalculator with /Users/shyue/.cache/mace/macempa0mediummodel
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64 for geometry optimization.
Using medium MPA-0 model as default MACE-MP model, to use previous (before 3.10) default model please specify 'medium' as model argument
Using Materials Project MACE for MACECalculator with /Users/shyue/.cache/mace/macempa0mediummodel
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64 for geometry optimization.


Using medium MPA-0 model as default MACE-MP model, to use previous (before 3.10) default model please specify 'medium' as model argument
Using Materials Project MACE for MACECalculator with /Users/shyue/.cache/mace/macempa0mediummodel
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64 for geometry optimization.


Running mp-22862 (NaCl):   9%|███████▉                                                                               | 1/11 [00:04<00:45,  4.51s/it]WARNING:root:Default dtype float32 does not match model dtype float64, converting models to float32.


Using medium MPA-0 model as default MACE-MP model, to use previous (before 3.10) default model please specify 'medium' as model argument
Using Materials Project MACE for MACECalculator with /Users/shyue/.cache/mace/macempa0mediummodel
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64 for geometry optimization.
Using medium MPA-0 model as default MACE-MP model, to use previous (before 3.10) default model please specify 'medium' as model argument
Using Materials Project MACE for MACECalculator with /Users/shyue/.cache/mace/macempa0mediummodel
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64 for geometry optimization.
Using medium MPA-0 model as default MACE-MP model, to use previous (before 3.10) default model please specify 'medium' as model argument
Using Materials Project MACE for MACECalculator with /Users/shyue/.cache/mace/macempa0mediummodel
Using float32 for MACECalculator, which

...

Running mp-1120767 (NaCl):  36%|██████████████████████████████▉                                                      | 4/11 [00:16<00:29,  4.26s/it]WARNING:root:Default dtype float32 does not match model dtype float64, converting models to float32.


Using medium MPA-0 model as default MACE-MP model, to use previous (before 3.10) default model please specify 'medium' as model argument
Using Materials Project MACE for MACECalculator with /Users/shyue/.cache/mace/macempa0mediummodel
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64 for geometry optimization.
Using medium MPA-0 model as default MACE-MP model, to use previous (before 3.10) default model please specify 'medium' as model argument
Using Materials Project MACE for MACECalculator with /Users/shyue/.cache/mace/macempa0mediummodel
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64 for geometry optimization.
Using medium MPA-0 model as default MACE-MP model, to use previous (before 3.10) default model please specify 'medium' as model argument
Using Materials Project MACE for MACECalculator with /Users/shyue/.cache/mace/macempa0mediummodel
Using float32 for MACECalculator, which

Running mp-755894 (Li2O):  45%|███████████████████████████████████████                                               | 5/11 [00:22<00:28,  4.70s/it]WARNING:root:Default dtype float32 does not match model dtype float64, converting models to float32.


Using medium MPA-0 model as default MACE-MP model, to use previous (before 3.10) default model please specify 'medium' as model argument
Using Materials Project MACE for MACECalculator with /Users/shyue/.cache/mace/macempa0mediummodel
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64 for geometry optimization.
Using medium MPA-0 model as default MACE-MP model, to use previous (before 3.10) default model please specify 'medium' as model argument
Using Materials Project MACE for MACECalculator with /Users/shyue/.cache/mace/macempa0mediummodel
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64 for geometry optimization.
Using medium MPA-0 model as default MACE-MP model, to use previous (before 3.10) default model please specify 'medium' as model argument
Using Materials Project MACE for MACECalculator with /Users/shyue/.cache/mace/macempa0mediummodel
Using float32 for MACECalculator, which

Running mp-1245224 (Li2O):  55%|██████████████████████████████████████████████▎                                      | 6/11 [00:37<00:41,  8.21s/it]

To make it easier to work with, we will convert our results into a pandas DataFrame.

In [ ]:
df_preds = pd.DataFrame(prop_preds)
df_preds["label"] = df_preds["material_id"] + "-" + df_preds["formula"]
df_preds.head()


## Plotting

We will now use plotly to make a few plots of the bulk modulus from ElasticityCalc and EOSCalc for comparisons.

In [ ]:
px.bar(data_frame=df_preds, x="label", y="bulk_modulus_vrh", color="model", barmode="group", title="ElasticityCalc")

In [ ]:
px.bar(data_frame=df_preds, x="label", y="bulk_modulus_bm", color="model", barmode="group", title="EOSCalc")

This is strange, but the EOSCalc results seem to make more sense. TensorNet-MatPES-PBE and MACE-MPA-0 are expected to yield similar results since they are based on the same PBE functional. The TensorNet-MatPES-r2SCAN results are expected to be different. The ElasticityCalc results show large differences between TensorNet-MatPES-PBE and MACE-MPA-0, but the EOSCalc shows much smaller differences. Let us compare the differences within the same model.

In [ ]:
df_preds["bulk_modulus_diff"] = (df_preds["bulk_modulus_vrh"] - df_preds["bulk_modulus_bm"]).abs()
px.bar(data_frame=df_preds, x="label", y="bulk_modulus_diff", color="model", barmode="group", title="Diff between Elasticity and EOS bulk modulus")

Quite clearly the MACE-MPA-0 shows much bigger inconsistencies for this limited subset of materials tested. This is not totally unexpected since the training dataset of MACE-MPA-0 is based on old MPRelax settings with poorer convergence settings. The MatPES dataset is a much more modern dataset with better convergence settings.

## EOS Curves

We will plot just one EOS curve as an illustration.

In [ ]:
eos = df_preds["eos"][0]

px.scatter(x=eos["volumes"], y=eos["energies"])